# SelectionExplainer — testing the selection shapes

Tests `explanation.inference.selection_explainer.SelectionExplainer` on the
same book used by the anti-spoiler feature (`antispoiler.book`, Pride and
Prejudice by default), reusing its `fetch_and_chunk()` pipeline so the text
comes from the exact same source/chunking as the spoiler mechanism.

The input can span **several paragraphs**; only the paragraph(s) actually
touched by the selection are ever sent to the LLM as context:

1. **Single word** — no local identification; the LLM defines it directly,
   grounded on the one paragraph the word sits in.
2. **Full sentence** — the trained Edit Predictor runs on that one sentence;
   context is the one paragraph containing it.
3. **Excerpt within one paragraph, crossing a sentence boundary** — the Edit
   Predictor runs on every full sentence the excerpt touches (even the part
   outside the excerpt), then only the difficult words inside the excerpt
   are kept and explained; context is that one paragraph.
4. **Excerpt crossing a paragraph boundary** — same as (3), except the
   touched sentences come from two different paragraphs, so both paragraphs
   are sent as context.

## 1. Setup

In [29]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    markers = [
        Path("explanation") / "inference" / "selection_explainer.py",
        Path("antispoiler") / "book.py",
    ]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError("Could not locate project root from " + str(start))


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/kenai/projects/responsible-ai-system-design


In [ ]:
import re
import textwrap

from explanation.inference.selection_explainer import SelectionExplainer

EDIT_PREDICTOR_CHECKPOINT = (
    PROJECT_ROOT
    / "explanation/model/checkpoints/edit_predictor_complex_words_distilbert_max256_big_final"
)
assert EDIT_PREDICTOR_CHECKPOINT.exists(), EDIT_PREDICTOR_CHECKPOINT

explainer = SelectionExplainer(
    edit_predictor_checkpoint=EDIT_PREDICTOR_CHECKPOINT,
    max_length=256,
)
print("Loaded SelectionExplainer with checkpoint:", EDIT_PREDICTOR_CHECKPOINT.name)


def show(result, title=None):
    """Pretty-print an ExplanationResult one line at a time (readable, not repr())."""
    if title:
        print(f"=== {title} ===")

    print("Context sent to the LLM:")
    for line in textwrap.wrap(result.sentence, width=90):
        print(f"  {line}")

    print()
    if not result.difficult_words:
        print("Difficult words: none")
        return

    print(f"Difficult words ({len(result.difficult_words)}):")
    for dw in result.difficult_words:
        print(f"  - {dw.word!r}  span={dw.span}")
        print(f"      {dw.meaning_in_context}")

## 2. Fetch the same book the anti-spoiler feature uses

`antispoiler.book.fetch_and_chunk` downloads the book and splits it into
`Chunk`s (one per paragraph, chapter-indexed) — the exact same pipeline the
anti-spoiler retrieval mechanism runs on. We reuse it here instead of a
separate hardcoded excerpt.

The raw Gutenberg text hard-wraps lines inside a paragraph (`\r\n`), which the
Edit Predictor never saw during training (it was trained on single-line
sentences) — so we normalize whitespace before using the text as our input.

In [ ]:
from antispoiler import config
from antispoiler.book import fetch_and_chunk


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def find_chunk(chunks, chunk_id):
    return next(c for c in chunks if c.chunk_id == chunk_id)


chunks = fetch_and_chunk(config.BOOK_URL)
print(f"Book: {config.BOOK_TITLE} by {config.BOOK_AUTHOR}")
print(f"Total chunks: {len(chunks)}")

opening_chunk = find_chunk(chunks, "ch01_p01")

multi_sentence_chunk = find_chunk(chunks, "ch04_p09")

print(f"\n{opening_chunk.chunk_id} ({opening_chunk.chapter_label}):")
print(repr(opening_chunk.text))
print(f"\n{multi_sentence_chunk.chunk_id} ({multi_sentence_chunk.chapter_label}):")
print(repr(multi_sentence_chunk.text))

Book: Pride and Prejudice by Jane Austen
Total chunks: 996

ch01_p01 (Chapter 1):
'It is a truth universally acknowledged, that a single man in possession\r\nof a good fortune, must be in want of a wife.\n\nHowever little known the feelings or views of such a man may be on his\r\nfirst entering a neighbourhood, this truth is so well fixed in the minds\r\nof the surrounding families, that he is considered the rightful property\r\nof some one or other of their daughters.\n\n"My dear Mr. Bennet," said his lady to him one day, "have you heard that\r\nNetherfield Park is let at last?"'

ch04_p09 (Chapter 4):
'Mrs. Hurst and her sister allowed it to be so--but still they admired\r\nher and liked her, and pronounced her to be a sweet girl, and one\r\nwhom they would not object to know more of. Miss Bennet was therefore\r\nestablished as a sweet girl, and their brother felt authorized by such\r\ncommendation to think of her as he chose.'


In [ ]:
from explanation.inference.selection_explainer import split_into_paragraphs

opening_sub_paragraphs = [p for p in opening_chunk.text.split("\n\n") if p.strip()]
para_1 = normalize_whitespace(opening_sub_paragraphs[0])
para_2 = normalize_whitespace(opening_sub_paragraphs[1])
para_3 = normalize_whitespace(multi_sentence_chunk.text)

text = "\n\n".join([para_1, para_2, para_3])

paragraphs = split_into_paragraphs(text)
assert [p.text for p in paragraphs] == [para_1, para_2, para_3]

for i, p in enumerate(paragraphs, start=1):
    print(f"--- paragraph {i} [{p.start},{p.end}) ---")
    print(textwrap.fill(p.text, width=90))
    print()

--- paragraph 1 [0,117) ---
It is a truth universally acknowledged, that a single man in possession of a good fortune,
must be in want of a wife.

--- paragraph 2 [119,376) ---
However little known the feelings or views of such a man may be on his first entering a
neighbourhood, this truth is so well fixed in the minds of the surrounding families, that
he is considered the rightful property of some one or other of their daughters.

--- paragraph 3 [378,696) ---
Mrs. Hurst and her sister allowed it to be so--but still they admired her and liked her,
and pronounced her to be a sweet girl, and one whom they would not object to know more of.
Miss Bennet was therefore established as a sweet girl, and their brother felt authorized
by such commendation to think of her as he chose.



## 3. Single-word selection

No local identification runs at all -- the word is already known, so the
Edit Predictor has nothing to add. `explain()` goes straight to the LLM,
grounded on just the one paragraph the word is in (paragraph 2 here) -- not
the whole three-paragraph `text`.

In [33]:
word_selection = "neighbourhood"
assert word_selection in para_2

result_word = explainer.explain(text, word_selection)
print("Context sent to the LLM == para_2 only:", result_word.sentence == para_2)
print()
show(result_word, title="Single word")

Context sent to the LLM == para_2 only: True

=== Single word ===
Context sent to the LLM:
  However little known the feelings or views of such a man may be on his first entering a
  neighbourhood, this truth is so well fixed in the minds of the surrounding families, that
  he is considered the rightful property of some one or other of their daughters.

Difficult words (1):
  - 'neighbourhood'  span=(88, 101)
      local area or community


## 4. Full-sentence selection

The selection matches one full sentence from paragraph 1 exactly, so the
trained Edit Predictor runs on it directly. Context is paragraph 1 alone.

In [34]:
from explanation.inference.selection_explainer import split_into_sentences

para_1_sentences = split_into_sentences(para_1)
for s in para_1_sentences:
    print(f"[{s.start:>3},{s.end:>3}) {s.text!r}")

sentence_selection = para_1_sentences[0].text
result_sentence = explainer.explain(text, sentence_selection)
print("Context sent to the LLM == para_1 only:", result_sentence.sentence == para_1)
print()
show(result_sentence, title="Full sentence")

[  0,117) 'It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.'
Context sent to the LLM == para_1 only: True

=== Full sentence ===
Context sent to the LLM:
  It is a truth universally acknowledged, that a single man in possession of a good fortune,
  must be in want of a wife.

Difficult words (4):
  - 'that'  span=(40, 44)
      introducing a clause
  - 'man'  span=(54, 57)
      adult male
  - 'good'  span=(77, 81)
      large or substantial
  - 'want'  span=(102, 106)
      lack or require


## 5. Excerpt within one paragraph, crossing a sentence boundary

`para_3` has two sentences. The excerpt below deliberately starts mid-way
through sentence 1 and ends mid-way through sentence 2 -- neither is fully
selected. The Edit Predictor still runs on both **full** sentences (using
paragraph 3's own sentence boundaries, not the excerpt's), and only the
difficult words whose span actually falls inside the excerpt are kept and
sent to the LLM. Context is paragraph 3 alone.

In [35]:
excerpt_start = text.index("sweet girl")  # first occurrence: inside para_3 sentence 1
excerpt_end = text.index("established") + len("established")  # inside para_3 sentence 2
excerpt_selection = text[excerpt_start:excerpt_end]

print("EXCERPT (crosses the sentence boundary within para_3):")
print(textwrap.fill(repr(excerpt_selection), width=90))

result_excerpt = explainer.explain(text, excerpt_selection)
print("\nContext sent to the LLM == para_3 only:", result_excerpt.sentence == para_3)
print()
show(result_excerpt, title="Excerpt within one paragraph")

EXCERPT (crosses the sentence boundary within para_3):
'sweet girl, and one whom they would not object to know more of. Miss Bennet was therefore
established'

Context sent to the LLM == para_3 only: True

=== Excerpt within one paragraph ===
Context sent to the LLM:
  Mrs. Hurst and her sister allowed it to be so--but still they admired her and liked her,
  and pronounced her to be a sweet girl, and one whom they would not object to know more of.
  Miss Bennet was therefore established as a sweet girl, and their brother felt authorized
  by such commendation to think of her as he chose.

Difficult words (4):
  - 'object'  span=(156, 162)
      have a dislike or opposition
  - 'know'  span=(166, 170)
      be familiar with
  - 'Miss'  span=(180, 184)
      title for unmarried woman
  - 'established'  span=(206, 217)
      generally recognized or accepted


## 6. Excerpt crossing a paragraph boundary

This excerpt starts inside paragraph 2's single sentence and ends inside
paragraph 3's first sentence -- it crosses the paragraph break entirely.
The Edit Predictor runs on both full touched sentences (one from each
paragraph), the result is filtered down to whatever falls inside the
excerpt, and **both paragraphs** (2 and 3) are joined and sent as context --
paragraph 1 is never touched and never sent.

In [36]:
crossing_start = text.index("surrounding families")  # inside para_2
crossing_end = text.index("admired her") + len("admired her")  # inside para_3
crossing_excerpt = text[crossing_start:crossing_end]

print("EXCERPT (crosses the paragraph boundary between para_2 and para_3):")
print(textwrap.fill(repr(crossing_excerpt), width=90))
assert "\n\n" in crossing_excerpt  # sanity check: it really crosses a paragraph break

result_crossing = explainer.explain(text, crossing_excerpt)
expected_context = para_2 + "\n\n" + para_3
print("\nContext sent to the LLM == para_2 + para_3:", result_crossing.sentence == expected_context)
print()
show(result_crossing, title="Excerpt crossing a paragraph boundary")

EXCERPT (crosses the paragraph boundary between para_2 and para_3):
'surrounding families, that he is considered the rightful property of some one or other of
their daughters.\n\nMrs. Hurst and her sister allowed it to be so--but still they admired
her'

Context sent to the LLM == para_2 + para_3: True

=== Excerpt crossing a paragraph boundary ===
Context sent to the LLM:
  However little known the feelings or views of such a man may be on his first entering a
  neighbourhood, this truth is so well fixed in the minds of the surrounding families, that
  he is considered the rightful property of some one or other of their daughters.  Mrs.
  Hurst and her sister allowed it to be so--but still they admired her and liked her, and
  pronounced her to be a sweet girl, and one whom they would not object to know more of.
  Miss Bennet was therefore established as a sweet girl, and their brother felt authorized
  by such commendation to think of her as he chose.

Difficult words (4):
  - 'surro

## 7. Summary

In [37]:
import pandas as pd

rows = []
for label, selection, result in [
    ("word", word_selection, result_word),
    ("sentence", sentence_selection, result_sentence),
    ("excerpt (1 paragraph)", excerpt_selection, result_excerpt),
    ("excerpt (crosses paragraphs)", crossing_excerpt, result_crossing),
]:
    for dw in result.difficult_words:
        rows.append({
            "selection_type": label,
            "selection": textwrap.shorten(selection, width=60),
            "context_paragraphs": result.sentence.count("\n\n") + 1,
            "word": dw.word,
            "span": dw.span,
            "meaning_in_context": dw.meaning_in_context,
        })

pd.DataFrame(rows)

,selection_type,selection,context_paragraphs,word,span,meaning_in_context
0,word,neighbourhood,1,neighbourhood,"(88, 101)",local area or community
1,sentence,"It is a truth universally acknowledged, that a...",1,that,"(40, 44)",introducing a clause
2,sentence,"It is a truth universally acknowledged, that a...",1,man,"(54, 57)",adult male
3,sentence,"It is a truth universally acknowledged, that a...",1,good,"(77, 81)",large or substantial
4,sentence,"It is a truth universally acknowledged, that a...",1,want,"(102, 106)",lack or require
5,excerpt (1 paragraph),"sweet girl, and one whom they would not object...",1,object,"(156, 162)",have a dislike or opposition
6,excerpt (1 paragraph),"sweet girl, and one whom they would not object...",1,know,"(166, 170)",be familiar with
7,excerpt (1 paragraph),"sweet girl, and one whom they would not object...",1,Miss,"(180, 184)",title for unmarried woman
8,excerpt (1 paragraph),"sweet girl, and one whom they would not object...",1,established,"(206, 217)",generally recognized or accepted
9,excerpt (crosses paragraphs),"surrounding families, that he is considered th...",2,surrounding,"(151, 162)",nearby or adjacent
